# 🎙️ Complete Self-Contained Speech Enhancement Training Pipeline
### Architecture: Conformer U-Net with Bounded Complex Ratio Masking (cRM)
This notebook contains the complete, self-contained code for dataset downloading, model architecture, loss functions, training loop, evaluation, and checkpoint downloading without requiring external file uploads.

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU
!nvidia-smi

# Install required dependencies
!pip install -q datasets soundfile matplotlib tqdm

## 2. Dataset Preparation (VoiceBank-DEMAND from Hugging Face)

In [ ]:
import os
import soundfile as sf
from datasets import load_dataset

print("📥 Fetching VoiceBank-DEMAND audio dataset...")
train_ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k", split="train[:2000]")
val_ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k", split="test[:200]")

os.makedirs("data/clean_train", exist_ok=True)
os.makedirs("data/noisy_train", exist_ok=True)
os.makedirs("data/clean_val", exist_ok=True)
os.makedirs("data/noisy_val", exist_ok=True)

print("💾 Saving training pairs to disk...")
for i, item in enumerate(train_ds):
    sf.write(f"data/clean_train/{i:05d}.wav", item["clean"]["array"], 16000)
    sf.write(f"data/noisy_train/{i:05d}.wav", item["noisy"]["array"], 16000)

print("💾 Saving validation pairs to disk...")
for i, item in enumerate(val_ds):
    sf.write(f"data/clean_val/{i:05d}.wav", item["clean"]["array"], 16000)
    sf.write(f"data/noisy_val/{i:05d}.wav", item["noisy"]["array"], 16000)

print(f"✅ Dataset preparation complete: {len(train_ds)} train pairs, {len(val_ds)} validation pairs.")

## 3. PyTorch Dataset, Conformer U-Net Model, Losses & Training Loop

In [ ]:
import os, glob, random, time
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- 1. DATASET ---
class AudioEnhancementDataset(Dataset):
    def __init__(self, clean_dir, noisy_dir, segment_seconds=2.0, is_train=True):
        self.clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
        self.noisy_dir = noisy_dir
        self.segment_len = int(segment_seconds * 16000)
        self.is_train = is_train

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        clean_path = self.clean_files[idx]
        noisy_path = os.path.join(self.noisy_dir, os.path.basename(clean_path))
        clean_audio, _ = sf.read(clean_path)
        noisy_audio, _ = sf.read(noisy_path)
        clean_audio = clean_audio.astype(np.float32)
        noisy_audio = noisy_audio.astype(np.float32)

        if len(clean_audio) > self.segment_len:
            start = random.randint(0, len(clean_audio) - self.segment_len) if self.is_train else 0
            clean_audio = clean_audio[start:start + self.segment_len]
            noisy_audio = noisy_audio[start:start + self.segment_len]
        else:
            pad = self.segment_len - len(clean_audio)
            clean_audio = np.pad(clean_audio, (0, pad))
            noisy_audio = np.pad(noisy_audio, (0, pad))

        return torch.from_numpy(noisy_audio), torch.from_numpy(clean_audio)

# --- 2. CONFORMER MODULES ---
class GLU(nn.Module):
    def __init__(self, dim=-1):
        super().__init__()
        self.dim = dim
    def forward(self, x):
        a, b = x.chunk(2, dim=self.dim)
        return a * torch.sigmoid(b)

class ConformerConvModule(nn.Module):
    def __init__(self, d_model=256, kernel_size=31, dropout=0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.pointwise_conv1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.glu = GLU(dim=1)
        self.depthwise_conv = nn.Conv1d(
            d_model, d_model, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, groups=d_model
        )
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.activation = nn.SiLU()
        self.pointwise_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (Batch, Time, Channels)
        residual = x
        x = self.layer_norm(x)
        x = x.transpose(1, 2)  # (B, C, T)
        x = self.pointwise_conv1(x)
        x = self.glu(x)
        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        x = self.pointwise_conv2(x)
        x = self.dropout(x)
        x = x.transpose(1, 2)  # (B, T, C)
        return residual + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, d_ff=1024, kernel_size=31, dropout=0.1):
        super().__init__()
        self.ffn1 = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_ff), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.mha_norm = nn.LayerNorm(d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.conv = ConformerConvModule(d_model=d_model, kernel_size=kernel_size, dropout=dropout)
        self.ffn2 = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_ff), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        norm_x = self.mha_norm(x)
        attn_out, _ = self.mha(norm_x, norm_x, norm_x)
        x = x + attn_out
        x = self.conv(x)
        x = x + 0.5 * self.ffn2(x)
        return self.final_norm(x)

# --- 3. CONFORMER U-NET ---
class ConformerUNet(nn.Module):
    def __init__(self, n_fft=512, hop_length=256, d_model=256, num_conformer_layers=4):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

        # Encoder (Frequency Downsampling)
        self.enc1 = nn.Sequential(nn.Conv2d(2, 32, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(32), nn.PReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(64), nn.PReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64, 128, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(128), nn.PReLU())
        self.enc4 = nn.Sequential(nn.Conv2d(128, 256, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(256), nn.PReLU())
        self.enc5 = nn.Sequential(nn.Conv2d(256, 256, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(256), nn.PReLU())

        # Bottleneck
        self.proj_in = nn.Linear(256 * 8, d_model)
        self.conformer_layers = nn.ModuleList([ConformerBlock(d_model=d_model) for _ in range(num_conformer_layers)])
        self.proj_out = nn.Linear(d_model, 256 * 8)

        # Decoder
        self.dec5 = nn.Sequential(nn.ConvTranspose2d(512, 256, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(256), nn.PReLU())
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(512, 128, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(128), nn.PReLU())
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(256, 64, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(64), nn.PReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(128, 32, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(32), nn.PReLU())
        self.dec1 = nn.ConvTranspose2d(64, 2, 3, stride=(2, 1), padding=1)

    def compute_stft(self, audio):
        stft = torch.stft(audio, self.n_fft, self.hop_length, win_length=self.n_fft, window=self.window, return_complex=True)
        return torch.cat([stft.real.unsqueeze(1), stft.imag.unsqueeze(1)], dim=1)

    def compute_istft(self, c_spec, length):
        complex_tensor = torch.complex(c_spec[:, 0], c_spec[:, 1])
        return torch.istft(complex_tensor, self.n_fft, self.hop_length, win_length=self.n_fft, window=self.window, length=length)

    def forward(self, audio_noisy):
        B, L = audio_noisy.shape
        spec_in = self.compute_stft(audio_noisy)
        x = spec_in[:, :, :256, :]
        T = x.shape[-1]

        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        feat = e5.permute(0, 3, 1, 2).contiguous().view(B, T, 256 * 8)
        feat = self.proj_in(feat)
        for layer in self.conformer_layers:
            feat = layer(feat)
        feat = self.proj_out(feat).view(B, T, 256, 8).permute(0, 2, 3, 1).contiguous()

        d5 = self.dec5(torch.cat([feat, e5], dim=1))
        d4 = self.dec4(torch.cat([d5, e4], dim=1))
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))

        mask_256 = torch.tanh(d1)
        mask = torch.cat([mask_256, torch.zeros(B, 2, 1, T, device=audio_noisy.device)], dim=2)
        
        Mr, Mi = mask[:, 0:1], mask[:, 1:2]
        Xr, Xi = spec_in[:, 0:1], spec_in[:, 1:2]
        
        Sr_hat = Mr * Xr - Mi * Xi
        Si_hat = Mr * Xi + Mi * Xr
        spec_out = torch.cat([Sr_hat, Si_hat], dim=1)

        audio_enh = self.compute_istft(spec_out, length=L)
        return audio_enh, spec_out

# --- 4. LOSS FUNCTIONS & METRICS ---
class HybridSpeechLoss(nn.Module):
    def __init__(self, gamma=0.3, alpha=0.5, lambda_time=0.05):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.lambda_time = lambda_time

    def forward(self, pred_audio, targ_audio, pred_spec, targ_spec):
        pr, pi = pred_spec[:, 0], pred_spec[:, 1]
        tr, ti = targ_spec[:, 0], targ_spec[:, 1]
        p_mag = torch.clamp(pr**2 + pi**2, min=1e-12)**(self.gamma / 2.0)
        t_mag = torch.clamp(tr**2 + ti**2, min=1e-12)**(self.gamma / 2.0)
        
        loss_mag = F.l1_loss(p_mag, t_mag)
        loss_c = F.l1_loss(pr * p_mag, tr * t_mag) + F.l1_loss(pi * p_mag, tr * t_mag)
        l_spec = self.alpha * loss_mag + (1.0 - self.alpha) * loss_c

        pred_z = pred_audio - torch.mean(pred_audio, dim=-1, keepdim=True)
        targ_z = targ_audio - torch.mean(targ_audio, dim=-1, keepdim=True)
        dot = torch.sum(pred_z * targ_z, dim=-1, keepdim=True)
        s_target = (dot / (torch.sum(targ_z**2, dim=-1, keepdim=True) + 1e-8)) * targ_z
        e_noise = pred_z - s_target
        si_snr = 10 * torch.log10(torch.sum(s_target**2, dim=-1) / (torch.sum(e_noise**2, dim=-1) + 1e-8))
        l_time = -torch.mean(si_snr)

        return l_spec + self.lambda_time * l_time, l_spec, l_time

def compute_si_sdr(estimated, reference, eps=1e-8):
    est = estimated - np.mean(estimated)
    ref = reference - np.mean(reference)
    s_target = (np.sum(est * ref) / (np.sum(ref ** 2) + eps)) * ref
    e_noise = est - s_target
    return float(10 * np.log10((np.sum(s_target ** 2) + eps) / (np.sum(e_noise ** 2) + eps)))

# --- 5. TRAINING EXECUTION LOOP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> Compute Device: {device}")

EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 5e-4
os.makedirs("./checkpoints", exist_ok=True)

train_loader = DataLoader(AudioEnhancementDataset("data/clean_train", "data/noisy_train", is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(AudioEnhancementDataset("data/clean_val", "data/noisy_val", is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = ConformerUNet().to(device)
criterion = HybridSpeechLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_gain = -float("inf")
print(f"--> Starting training on {len(train_loader.dataset)} audio samples for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            targ_spec = model.compute_stft(clean)
            pred_audio, pred_spec = model(noisy)
            loss, _, _ = criterion(pred_audio, clean, pred_spec, targ_spec)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    sdr_gains = []
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            pred_audio, _ = model(noisy)
            for p, c, n in zip(pred_audio.cpu().numpy(), clean.cpu().numpy(), noisy.cpu().numpy()):
                sdr_gains.append(compute_si_sdr(p, c) - compute_si_sdr(n, c))

    avg_sdr_gain = float(np.mean(sdr_gains)) if sdr_gains else 0.0
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} ({time.time()-t0:.1f}s) | Val ΔSI-SDR: +{avg_sdr_gain:.2f} dB")

    if avg_sdr_gain > best_gain:
        best_gain = avg_sdr_gain
        torch.save(model.state_dict(), "./checkpoints/best_model.pth")
        print(f"  ⭐ New best checkpoint saved (+{best_gain:.2f} dB) to ./checkpoints/best_model.pth")

print("\n🎉 Training finished successfully!")

## 4. Download `best_model.pth`

In [ ]:
from google.colab import files
files.download("./checkpoints/best_model.pth")